# PAD-UFES-20 Image-Only Baseline

Run this notebook in Google Colab with a GPU runtime. It prepares the PAD-UFES-20 dataset, creates patient-safe splits, then launches the reusable training CLI in `src.training.train_image_baseline`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os
import subprocess

try:
    from google.colab import userdata
except ImportError:
    userdata = None


def get_config(name, default=None):
    value = os.environ.get(name)
    if value:
        return value
    if userdata is not None:
        try:
            return userdata.get(name) or default
        except Exception:
            return default
    return default


def export_config(name, default=None, required=False):
    value = get_config(name, default)
    if required and not value:
        raise RuntimeError(f'Set {name} in Colab Secrets before training.')
    if value:
        os.environ[name] = str(value)
    return value


REPO_URL = 'https://github.com/SalmaneSossey/mlops-teledermatology.git'
BRANCH = 'main'
REPO_DIR = Path('/content/mlops-teledermatology')
HF_DATASET_REPO = get_config('PAD_UFES20_HF_REPO_ID', 'SalmaneExploring/pad-ufes-20')
DATA_ROOT = Path('/content/pad_ufes_20')
IMAGES_DIR = DATA_ROOT / 'all_images'
OUTPUT_DIR = Path('/content/drive/MyDrive/mlops-teledermatology/runs/image_baseline')

export_config('DAGSHUB_TOKEN', required=True)
export_config('DAGSHUB_USERNAME')
export_config('DAGSHUB_REPO_OWNER', 'SalmaneSossey')
export_config('DAGSHUB_REPO_NAME', 'mlops-teledermatology')
export_config('DAGSHUB_MLFLOW_TRACKING_URI')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    subprocess.run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], cwd='/content', check=True)

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())
print('Hugging Face dataset:', HF_DATASET_REPO)
print('PAD-UFES-20 data root:', DATA_ROOT)
print('PAD-UFES-20 images dir:', IMAGES_DIR)
print('MLflow tracking URI:', os.environ.get('DAGSHUB_MLFLOW_TRACKING_URI') or f"https://dagshub.com/{os.environ['DAGSHUB_REPO_OWNER']}/{os.environ['DAGSHUB_REPO_NAME']}.mlflow")


In [ ]:
!pip -q install mlflow huggingface_hub


In [ ]:
!python -m src.data.download_pad_ufes_20 \
  --repo-id "{HF_DATASET_REPO}" \
  --output-dir "{DATA_ROOT}" \
  --force

!python -m src.data.make_image_splits \
  --metadata-path "{DATA_ROOT / 'metadata.csv'}" \
  --images-dir "{IMAGES_DIR}" \
  --output-dir data/processed/splits


In [ ]:
!python -m src.training.train_image_baseline \
  --images-dir "{IMAGES_DIR}" \
  --splits-dir data/processed/splits \
  --output-dir "{OUTPUT_DIR}" \
  --hf-dataset-repo "{HF_DATASET_REPO}"
